# BA Research Qev — local enterprise decision template

This notebook demonstrates Qev as an experimental typed-decision scorer, not a generative assistant or autonomous operator. It includes fictional oil & gas and general enterprise examples. Every example is synthetic; do not paste operational, personal, confidential, export-controlled, or safety-critical data.

> **Important deployment boundary:** Google Colab is a Google-hosted cloud runtime, not offline, private infrastructure, or a self-hosted enterprise environment. Notebook code below deliberately does not download packages, model weights, contact a provider, upload data, or publish anything. Running real inference requires dependencies and the complete, rights-authorized Qev checkpoint plus its exact pinned Qwen base to have been provisioned locally already. If they are absent, inference stops with a clear error. The Qev family remains restricted to local/internal use pending rights review.

To use this as an actual local/offline template, download/provision dependencies and model files only through your organization's approved process on an authorized local machine, verify licenses and hashes, then run the notebook with network disabled. Keep all paths and outputs in approved local storage. Do not use Colab for restricted inputs or to transfer internal checkpoints.

## What this demonstrates (and does not)

- The supported Open-Jev typed interface: `compile_request` → `DecisionModel.load` → scores → `format_response`. Qev scores candidate options; it does not explain or execute the selected action.
- Two fictional cases: an oilfield maintenance triage decision and a vendor-payment control decision. A third example illustrates a lower-risk document task.
- A deterministic, offline fixture mode that validates request construction without pretending to perform inference.
- Explicit separation between model scores, human operational judgment, and any downstream workflow.

Not demonstrated: Jev parity, production reliability, calibrated confidence for these scenarios, actual safety effectiveness, or a complete enterprise deployment. The training acceptance reports experimental validation results and explicitly does not establish fresh test/OOD quality, production readiness, or Jev parity. Oil & gas scenarios here are illustrative, not domain validation.

In [ ]:
from pathlib import Path
import json
import os

# Set these only to pre-provisioned local paths. No download or install is attempted.
CHECKPOINT_DIR = Path(os.environ.get('QEV_CHECKPOINT_DIR', './qev-checkpoint')).expanduser()
BASE_MODEL_DIR = Path(os.environ.get('QEV_BASE_MODEL_DIR', './Qwen3.5-0.8B')).expanduser()
RUN_REAL_INFERENCE = False  # Opt in only after authorized local provisioning.

print('Checkpoint:', CHECKPOINT_DIR.resolve())
print('Pinned base:', BASE_MODEL_DIR.resolve())
print('Execution mode:', 'real model' if RUN_REAL_INFERENCE else 'offline fixture only')

## Synthetic scenarios

Choice labels are candidate identifiers, not confidence categories. Edit the fictional wording and options for a research exercise, then have domain and safety owners review the decision framing before any pilot. Avoid using Qev as the sole basis for a field, maintenance, process-safety, financial-control, or compliance decision.

In [ ]:
CASES = {
    'oil_gas_maintenance': {
        'state': 'Fictional site: a vibration trend on a non-critical pump is above its normal band. No alarm threshold has been crossed, operations are stable, and the next planned inspection is in 72 hours. No real asset or site data.',
        'questions': {'next_step': {
            'type': 'choice',
            'instructions': 'Select the most prudent administrative next step for human review; this is not an operating instruction.',
            'criteria': {
                'A': 'Continue normal monitoring and document the trend for the planned inspection.',
                'B': 'Request qualified maintenance review and check the approved site procedure before deciding whether to advance inspection.',
                'C': 'Immediately shut down the equipment based only on this summary.',
            },
        }},
    },
    'vendor_payment_control': {
        'state': 'Fictional back-office case: a vendor requests a bank-account change shortly before payment. The request has not been independently verified. No real account, vendor, or payment data.',
        'questions': {'next_step': {
            'type': 'choice',
            'instructions': 'Choose a control-preserving administrative next step for a human reviewer.',
            'criteria': {
                'A': 'Pause the change and verify it through the established independent channel.',
                'B': 'Update the account using the request email and process payment.',
                'C': 'Reject all future payments to this vendor.',
            },
        }},
    },
    'document_routing': {
        'state': 'Fictional shared-services request: correct a typo in an internal procedure index; no policy meaning changes.',
        'questions': {'route': {
            'type': 'choice',
            'instructions': 'Choose the appropriate review lane for this described work.',
            'criteria': {
                'A': 'Routine clerical correction with a reviewer checking the diff.',
                'B': 'Cross-team policy redesign requiring specialist review.',
                'C': 'Emergency incident response.',
            },
        }},
    },
}

assert len(CASES) == 3 and all('state' in case and 'questions' in case for case in CASES.values())
[(name, len(case['state']), list(case['questions'])) for name, case in CASES.items()]

## Offline fixture: validate only the authored requests

This cell checks that each request is accepted by the interface compiler when Open-Jev is available. It never loads a model or emits a Qev decision. If the interface package is not pre-installed, it reports that dependency as missing.

In [ ]:
try:
    from jev.api import compile_request
except ImportError as exc:
    compile_request = None
    print(f'Open-Jev interface unavailable; request fixture not run: {exc}')

compiled_cases = {}
if compile_request is not None:
    for name, request in CASES.items():
        compiled = compile_request(request['state'], request['questions'])
        assert compiled, f'{name}: compiler returned no records'
        compiled_cases[name] = compiled
        print(f'{name}: valid typed request ({len(compiled)} question record(s)); NO inference performed')
    assert set(compiled_cases) == set(CASES)
    print('OFFLINE FIXTURE PASS: request shape only; no model loaded, network call or decision made.')

## Optional real inference — local files only

The following cell runs only if you explicitly set `RUN_REAL_INFERENCE = True` and both local directories exist. The checkpoint must be the accepted Open-Jev-compatible Qev **general-purpose** checkpoint layout (adapter, `head.pt`, `model.json`, `temperature.json`); the base must be the exact revision named in `model.json`. This cell points `model_id` at the supplied local base path and forces Hugging Face offline mode. Missing files, incompatible model/code versions, or runtime failures are raised visibly—there is no fallback to a hosted model.

Use your environment's already-installed, validated versions of PyTorch, Transformers, PEFT, Accelerate, Hugging Face Hub, and the exact pinned Open-Jev source revision recorded by the project. For a cloud runtime, dependencies can be installed deliberately using the pinned commands in the repository local/requirements.txt; this requires internet and is not an offline workflow. The requirements file is not a guarantee of Colab compatibility. This notebook does not install packages automatically; provision dependencies explicitly in an approved environment. If your approved local runtime cannot satisfy these prerequisites, stop here and report the error rather than changing models or silently using a service.

In [ ]:
if not RUN_REAL_INFERENCE:
    print('SKIPPED: set RUN_REAL_INFERENCE=True only in an authorized local runtime with pre-provisioned files.')
else:
    required = [
        CHECKPOINT_DIR / 'model.json',
        CHECKPOINT_DIR / 'head.pt',
        CHECKPOINT_DIR / 'temperature.json',
        CHECKPOINT_DIR / 'adapter' / 'adapter_config.json',
        CHECKPOINT_DIR / 'adapter' / 'adapter_model.safetensors',
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError('Qev checkpoint is incomplete; no inference performed. Missing: ' + ', '.join(missing))
    if not BASE_MODEL_DIR.is_dir():
        raise FileNotFoundError(f'Pinned local Qwen base directory is missing: {BASE_MODEL_DIR}; no download attempted.')
    model_meta = json.loads((CHECKPOINT_DIR / 'model.json').read_text(encoding='utf-8'))
    temperature_meta = json.loads((CHECKPOINT_DIR / 'temperature.json').read_text(encoding='utf-8'))
    expected_revision = model_meta.get('revision')
    if not expected_revision or not model_meta.get('model_id'):
        raise ValueError('Checkpoint model.json lacks base model identity/revision; refusing inference.')
    temperature = temperature_meta.get('temperature')
    if temperature_meta.get('split') != 'calibration' or not isinstance(temperature, (int, float)) or temperature <= 0:
        raise ValueError('Checkpoint calibration metadata is missing or invalid; refusing inference.')
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    from jev.model import DecisionModel
    from jev.api import format_response
    import torch
    from pathlib import Path as _Path
    # Require an explicit local provisioning assertion; this notebook cannot
    # infer revision identity from config.json or a directory name.
    provision_manifest = BASE_MODEL_DIR / 'qev-provision.json'
    if not provision_manifest.is_file():
        raise FileNotFoundError(f'Missing local provisioning assertion: {provision_manifest}; refusing to assume base revision.')
    provision = json.loads(provision_manifest.read_text(encoding='utf-8'))
    if provision.get('revision') != expected_revision:
        raise ValueError(f'Local base revision assertion mismatch: expected {expected_revision}, got {provision.get("revision")!r}')
    # DecisionModel.load reads model.json. Copy the checkpoint and change
    # only model_id to this asserted local base path; adapter and head files are copied into a temporary directory.
    if not (BASE_MODEL_DIR / 'config.json').is_file():
        raise FileNotFoundError(f'Local base lacks config.json: {BASE_MODEL_DIR}')
    # Open-Jev loads the base identity from checkpoint metadata. Use a private temporary
    # metadata copy outside the checkpoint so its model_id resolves to the local base.
    import tempfile, shutil
    local_checkpoint = Path(tempfile.mkdtemp(prefix='qev-local-checkpoint-')) / 'checkpoint'
    shutil.copytree(CHECKPOINT_DIR, local_checkpoint)
    local_model_meta = json.loads((local_checkpoint / 'model.json').read_text(encoding='utf-8'))
    local_model_meta['model_id'] = str(BASE_MODEL_DIR.resolve())
    (local_checkpoint / 'model.json').write_text(json.dumps(local_model_meta), encoding='utf-8')
    model = DecisionModel.load(local_checkpoint, device='cuda:0' if torch.cuda.is_available() else 'cpu')
    results = {}
    with torch.inference_mode():
        for name, request in CASES.items():
            records = compile_request(request['state'], request['questions'])
            logits = model(records)
            probabilities = [torch.softmax(row.double() / float(temperature), dim=-1).cpu().tolist() for row in logits]
            results[name] = format_response(records, probabilities)
    for name, result in results.items():
        print(f'\n{name}: {json.dumps(result, ensure_ascii=False, indent=2)}')
    print('Inference completed locally. Treat outputs as unvalidated research signals for human review, not operational recommendations.')

## Enterprise/local deployment checklist

1. Confirm the intended task, users, affected people/assets, and a named accountable human decision owner.
2. Review source, adapter, base-model, code, and deployment rights independently; Qev remains local/internal until the outstanding review is resolved.
3. Provision the exact adapter + decision head + calibration metadata + pinned Qwen base through approved internal channels. Verify recorded hashes and base revision; never upload the restricted bundle.
4. Install the validated dependency lock in a controlled environment before disconnecting networking. This notebook intentionally does not bootstrap dependencies.
5. Exercise representative, human-reviewed and appropriately rights-cleared examples; measure errors by decision type and operating context. Synthetic examples here are not evaluation data.
6. Add access control, audit logging, retention limits, resource monitoring, model/version pinning, rollback, and a tested human override in the host application. Those controls are outside this notebook.
7. For oil & gas or other safety-related workflows, use approved procedures and qualified personnel. Do not use this experimental model to control equipment, approve work, or replace safety systems.

### Parent-run real-model verification (required before claiming inference works)

On an authorized local machine—not Colab—identify an already-provisioned accepted checkpoint directory and matching base directory at the exact pinned revision in its `model.json`. Confirm the validated Open-Jev checkout and dependencies are installed in the active interpreter. Set `QEV_CHECKPOINT_DIR` and `QEV_BASE_MODEL_DIR`, restart the kernel with network disabled, set `RUN_REAL_INFERENCE = True`, and run all cells. Record the checkpoint/base hashes, interpreter and package versions, device, output, latency, and any error in a private, access-controlled record; do not copy or upload artifacts. Success requires all three requests to compile and return finite, normalized option probabilities through `DecisionModel.load` and Open-Jev `format_response`, with no network access. A fixture pass alone is not a model-runtime pass. Review outputs with domain owners; do not claim quality or operational readiness from this smoke test.